In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)
!ls $path

In [ ]:
# Task 1: Write your code here:
import pandas as pd, os
df = pd.read_csv(os.path.join(path, 'Q3_data.csv'))

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
most_null = df.isnull().sum().sort_values(ascending=False)[:34]
most_null

In [ ]:
# Task 1 CONTD.
# I sorted in above to see the 34 columns with most missing values. (descending order)
# table has 20000 entries. many cols like D_87 etc have almost all missing values (19k+).
# hence useless. better to drop these features.
# for now i will make a list of features which are missing in more than 25% of data (i.e. 5000+ null entries)
# and drop those.
# > 5000 entries are missing in the aforementioned 34 cols only.

most_null_cols = most_null.index
most_null_cols
df = df.drop(most_null_cols, axis=1)
df

In [ ]:
# Task 2: Write your code here:
df.duplicated().sum() # no dupes. (output = np.int64(0)) so no need to drop.

In [ ]:
# Task 3: Write your code here:
catcols = []
for col in df.columns:
  if df[col].dtype != 'float64' and df[col].dtype != 'int64':
    catcols.append((col, df[col].dtype))
print(catcols)
# since too many cols to see manually with df.info(), did this. :D
# no categorical values! :D

In [ ]:
features = df.drop(['Target'], axis=1).columns
for col in df.columns:
  df[col] = df[col].fillna(df[col].mean()) # since all are numeric. imputing with mean in the interest of time.

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

scale = StandardScaler()

df[features] = scale.fit_transform(df[features]) # not scaling target
df

In [ ]:
df.isnull().sum().sum() # just crosscheck. aha! zero nulls.

In [ ]:
# Task 5: Write your code here:
df['Target'].value_counts().plot(kind='bar')

import matplotlib.pyplot as plt
plt.title("Target Distro")
plt.ylabel("Clearly imbalanced.")
# clearly target is imbalanced. 0 is ~3x as frequent as 1.

In [ ]:
# Task 1: Write your code here:
X = df.drop(['Target'], axis=1)
y = df['Target']
X

In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:

# StratifiedKFold correct since:
# - classification problem, and
# - class imbalance.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

import numpy as np

f1s = []

cb = CatBoostClassifier(verbose=0)


for tr_idx, val_idx in skf.split(X, y):
    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    cb.fit(X_tr, y_tr)
    y_pred = cb.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    f1s.append(f1)
    print("FOLD DONE.") # for debug. took way too long to train.
print(f"Avg F1 across all folds: {np.mean(f1s):.4f}")

In [ ]:
# Task 1: Write your code here:
fimps = pd.Series(cb.feature_importances_, index=X.columns)
fimps

In [ ]:
fimps.sort_values(ascending=False)[:10].plot(kind='barh')
plt.gca().invert_yaxis()
plt.title("Feature importances of CatBoost Model (Top 10 only)")

In [ ]:
# Task 2: Write your code here:
golden_feature = fimps.sort_values(ascending=False).index[0]
print("The most important feature is: (Golden Feature):", golden_feature)
# CHECK NEXT CELLS: BONUS ATTEMPTED!!

In [ ]:
# Task Bonus: Write your code here:

# note: you're saying in pt. 3. compare accuracy with model. but there you said choose correct metric; f1 score or acc.
# so should i compare acc or f1 score now? i will presume f1 score, since that is already collected for previous model.
# i don't want to waste time training it again and calc'ing acc now; takes ~5m to train 5 folds.

X_golden = pd.DataFrame(X[golden_feature])
X_golden

In [ ]:
cb_golden = CatBoostClassifier(verbose=0)


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

import numpy as np

f1s_golden = []

cb_golden = CatBoostClassifier(verbose=0)


for tr_idx, val_idx in skf.split(X_golden, y):
    X_tr = X_golden.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]

    X_val = X_golden.iloc[val_idx]
    y_val = y.iloc[val_idx]

    cb_golden.fit(X_tr, y_tr)
    y_pred = cb_golden.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    f1s_golden.append(f1)
    print("FOLD DONE.") # for debug. took way too long to train.
print(f"Avg F1 across all folds (GOLDEN): {np.mean(f1s_golden):.4f}")

In [ ]:
import seaborn as sns
sns.set_style('whitegrid')
plt.plot(f1s, label='All Features')
plt.plot(f1s_golden, label = 'Golden Feature Only')
plt.plot([0, 4], [1, 1], "r--", label='Perfect F1 Score (1)')
plt.legend()
plt.title("F1 Score Comparison for Models")
plt.xlabel('Fold Number (From 0 to 4).')
plt.ylim(-0.2, 1.2)
# Comparable performance! :DDDDD

In [ ]:
# thank you for your time and effort.